# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

from src.load_images import prepare_image_column, display_artwork_image
from src.gan_trainer import ArtGANTrainer

import torch
from torch.utils.data import TensorDataset, DataLoader
from torchvision.utils import make_grid

# Data

In [2]:
df = pd.read_csv('data/art_catalog.xlsx - catalog.csv')

In [3]:
df.head(2)

,AUTHOR,BORN-DIED,TITLE,DATE,TECHNIQUE,LOCATION,URL,FORM,TYPE,SCHOOL,TIMEFRAME
0,"AACHEN, Hans von","(b. 1552, Köln, d. 1615, Praha)",Venus and Adonis,1574-88,"Oil on canvas, 68 x 95 cm","Fogg Art Museum, Harvard University, Cambridge",https://www.wga.hu/html/a/aachen/adonis.html,painting,mythological,German,1601-1650
1,"AACHEN, Hans von","(b. 1552, Köln, d. 1615, Praha)",Allegory,1598,"Oil on copper, 56 x 47 cm","Alte Pinakothek, Munich",https://www.wga.hu/html/a/aachen/allegory.html,painting,mythological,German,1601-1650


In [4]:
painting_df = df[df['FORM'] == 'painting']
painting_df.head(2)

,AUTHOR,BORN-DIED,TITLE,DATE,TECHNIQUE,LOCATION,URL,FORM,TYPE,SCHOOL,TIMEFRAME
0,"AACHEN, Hans von","(b. 1552, Köln, d. 1615, Praha)",Venus and Adonis,1574-88,"Oil on canvas, 68 x 95 cm","Fogg Art Museum, Harvard University, Cambridge",https://www.wga.hu/html/a/aachen/adonis.html,painting,mythological,German,1601-1650
1,"AACHEN, Hans von","(b. 1552, Köln, d. 1615, Praha)",Allegory,1598,"Oil on copper, 56 x 47 cm","Alte Pinakothek, Munich",https://www.wga.hu/html/a/aachen/allegory.html,painting,mythological,German,1601-1650


In [5]:
painting_df = painting_df.drop(columns=['DATE', 'BORN-DIED', 'LOCATION', 'TECHNIQUE', 'URL', 'SCHOOL', 'TIMEFRAME'])

In [6]:
print(painting_df.shape)
painting_df.head(2)

(32438, 4)


,AUTHOR,TITLE,FORM,TYPE
0,"AACHEN, Hans von",Venus and Adonis,painting,mythological
1,"AACHEN, Hans von",Allegory,painting,mythological


In [7]:
painting_df.TYPE.value_counts(normalize=True)

TYPE
religious       0.416703
portrait        0.148283
landscape       0.126487
mythological    0.098711
genre           0.085455
still-life      0.043313
other           0.029903
historical      0.029071
interior        0.019607
study           0.002466
Name: proportion, dtype: float64

In [8]:
painting_df.TYPE.value_counts()

TYPE
religious       13517
portrait         4810
landscape        4103
mythological     3202
genre            2772
still-life       1405
other             970
historical        943
interior          636
study              80
Name: count, dtype: int64

In [9]:
painting_df.to_csv('data/full_painting_catalog.csv', index=False)

In [10]:
painting_df.iloc[0]

AUTHOR    AACHEN, Hans von
TITLE     Venus and Adonis
FORM              painting
TYPE          mythological
Name: 0, dtype: object

In [11]:
choosen_types_df = painting_df[painting_df['TYPE'].isin(['religious', 'portrait', 'landscape', 'mythological'])]
print(choosen_types_df.TYPE.value_counts())


TYPE
religious       13517
portrait         4810
landscape        4103
mythological     3202
Name: count, dtype: int64


In [12]:

choosen_types_df.to_csv('data/choosen_types_painting_catalog.csv', index=False)

## models

### training on WGAN-GP + résidus

#### Ici tu peux éventuellement changer p en 128 mais ça va être d'autant plus long

In [13]:
p = "64"
processed_painting_df = prepare_image_column(painting_df, "IMAGE", p)
processed_painting_df.head(2)
processed_painting_df.shape
processed_painting_df.to_csv(f'data/{p}image_catalog.csv', index=False)

Transform '64' processed 31062/32438 images.
Dropped 1376 rows due to missing or unreadable images.


In [14]:
images = torch.from_numpy(
    np.stack(processed_painting_df["IMAGE"].to_numpy())
).permute(0, 3, 1, 2)            # (N, C, H, W) = (N, 3, 64, 64)

dataset = TensorDataset(images)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

In [15]:
wgan_trainer = ArtGANTrainer(
      mode="excellent",
      image_size=64,
      early_stopping_metric="fid",
      early_stopping_patience=10,
      fid_every=1,
      fid_samples=256,
)

/Users/julienrm/.pyenv/versions/ganisme/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


#### Ici tu choisis le nombre d'epochs et le learning rate

In [16]:
history = wgan_trainer.train(dataloader, epochs=50, lr=2e-4)
wgan_trainer.plot_history()


KeyboardInterrupt: 

In [ ]:
samples = wgan_trainer.sample(16).cpu()
grid = (samples + 1) / 2  # back to [0,1] for matplotlib

# samples = wgan_trainer.sample(16).cpu()
# grid = (samples + 1) / 2  # back to [0,1] for display
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(make_grid(grid, nrow=4).permute(1, 2, 0).numpy())
ax.axis("off")
plt.show()

plt.savefig(f'output_{p}.png', bbox_inches='tight', dpi=150)